![Banner](../Image/03_DeepCuaslaML.png)

# 4.2 Structural Causal Models (SCMs) with Deep Components



## Part 1: Theory

An SCM is $\mathcal{M}=(\mathbf{S}, P(\mathbf{U}))$ with structural equations

$$X_i = f_i(\mathrm{Pa}(X_i), U_i).$$

SCMs support Pearl's hierarchy:
- Association: $P(Y\mid X=x)$
- Intervention: $P(Y\mid do(X=x))$
- Counterfactual: $P(Y_x\mid X=x', Y=y')$

Deep SCMs replace hand-crafted $f_i$ with neural nets:

$$X_i = f_{\theta_i}(\mathrm{Pa}(X_i), U_i).$$

Variational Deep SCM setup:
- Encoder: $q_\phi(U\mid X)$
- Decoder: $p_\theta(X\mid U,G)$
- ELBO-style objective with reconstruction + KL

DECI jointly learns graph and structural equations with NOTEARS penalty:

$$h(\mathbf{A}) = \mathrm{tr}(e^{\mathbf{A}\odot\mathbf{A}})-d.$$

## Setup and Imports

This cell imports the required libraries, sets random seeds for reproducibility, and selects the compute device (`CPU` or `CUDA`).

In [ ]:
import importlib
import subprocess
import sys

PACKAGES = [
    "numpy", "pandas", "scipy", "torch", "scikit-learn",
    "matplotlib", "seaborn", "networkx", "yfinance",
]

for pkg in PACKAGES:
    mod = "sklearn" if pkg == "scikit-learn" else pkg
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    import pydeepcausalml  # noqa: F401
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/zia207/PyDeepCausalML.git"]
    )

import pydeepcausalml
print("pydeepcausalml", pydeepcausalml.__version__, "ready.")


import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from pydeepcausalml import set_seed

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())

set_seed(42)
run_fast = True


import networkx as nx
from pydeepcausalml import DeepSCM, DECI, DynoTEARS
from pydeepcausalml.metrics import graph_recovery_metrics


## Data Loading and Preprocessing

This cell downloads sector ETF prices, converts them to standardized log-returns, builds lagged input/output pairs, and creates training/validation data loaders.

*Note: If online market data is unavailable, this notebook automatically uses synthetic demo data so all downstream cells remain runnable.*

In [ ]:
TICKERS = {
    "XLF": "Financials",
    "XLE": "Energy",
    "XLK": "Technology",
    "XLV": "Healthcare",
    "XLI": "Industrials",
    "XLY": "ConsumerDisc",
    "XLP": "ConsumerStap",
    "XLU": "Utilities",
}

import yfinance as yf


def fetch_close_prices(tickers, start="2018-01-01", end="2024-01-01"):
    raw = yf.download(
        tickers, start=start, end=end, auto_adjust=True,
        progress=False, group_by="ticker", threads=False,
    )
    close = None
    if isinstance(raw.columns, pd.MultiIndex):
        if "Close" in raw.columns.get_level_values(-1):
            close = raw.xs("Close", axis=1, level=-1)
    elif "Close" in raw.columns:
        close = raw[["Close"]]
        close.columns = tickers[:1]

    if close is None or close.dropna(how="all").empty:
        cols = []
        for t in tickers:
            one = yf.download(t, start=start, end=end, auto_adjust=True, progress=False, threads=False)
            if "Close" in one.columns and not one["Close"].dropna().empty:
                cols.append(one["Close"].rename(t))
        close = pd.concat(cols, axis=1) if cols else pd.DataFrame()
    return close.sort_index().dropna(how="all")


close = fetch_close_prices(list(TICKERS.keys()))
if close.empty or close.shape[0] < 50:
    print("yfinance unavailable; using synthetic demo data.")
    rng = np.random.default_rng(42)
    T, d = 1500, len(TICKERS)
    VAR_NAMES = list(TICKERS.values())
    market = rng.normal(0.0, 0.8, size=(T, 1)).astype(np.float64)
    idio = rng.normal(0.0, 0.6, size=(T, d)).astype(np.float64)
    loading = np.linspace(0.5, 1.1, d, dtype=np.float64)[None, :]
    data_np = (market @ loading + idio).astype(np.float64)
else:
    returns = np.log(close / close.shift(1)).dropna()
    returns.columns = [TICKERS[t] for t in returns.columns if t in TICKERS]
    data_np = returns.values.astype(np.float64)
    T, d = data_np.shape
    VAR_NAMES = returns.columns.tolist()

print(f"Shape: {data_np.shape}")
print(f"Variables ({d}): {VAR_NAMES}")

LAG = 1
EPOCHS_SCM = 25 if run_fast else 50
EPOCHS_DECI = 40 if run_fast else 80
EPOCHS_DYNO = 80 if run_fast else 200
DEVICE = "cpu"


## Utilities and Model Definitions (DeepSCM + DECI)

This section defines helper functions (acyclicity penalty, graph thresholding, plotting) and introduces the three structural models used in this notebook.

### 1) DeepSCM (Fixed-Graph Structural Model)

**DeepSCM** assumes a known adjacency matrix and learns nonlinear structural equations on top of that fixed graph.

A simplified form is:

$$
X_t = f_\theta\big(X_{t-1:t-p}, A\big) + \epsilon_t,
$$

where $A$ is fixed (not learned), and $f_\theta$ is represented by neural components.

Why useful:
- Stable training when a credible prior graph is available.
- Good baseline for intervention-style analysis.

Limitation:
- Performance and interpretability depend on the quality of the fixed graph.

### 2) DECI (Deep End-to-end Causal Inference)

**DECI** jointly learns both graph structure and structural equations in a continuous optimization framework.

It uses a soft adjacency matrix and enforces acyclicity with a NOTEARS-style constraint:

$$
h(A) = \operatorname{tr}(e^{A \circ A}) - d.
$$

A common objective combines reconstruction, sparsity, and DAG regularization:

$$
\mathcal{L} = \mathcal{L}_{\text{recon}} + \lambda_1\|A\|_1 + \lambda_2 h(A).
$$

Why useful:
- Learns structure and mechanism together.
- Produces interpretable weighted causal graphs.

Limitation:
- Hyperparameter-sensitive and can be optimization-heavy.

### 3) DYNOTEARS (Lagged DAG-Constrained Discovery)

**DYNOTEARS** extends NOTEARS-style optimization to time-series causal discovery with lagged dependencies.

For lag order $p$, a common formulation is:

$$
X_t = W^{(0)}X_t + \sum_{k=1}^{p} W^{(k)}X_{t-k} + \epsilon_t,
$$

where:
- $W^{(0)}$ captures same-time effects,
- $W^{(k)}$ captures directed lag-$k$ effects,
- acyclicity is typically imposed on the instantaneous component (or an aggregated proxy) via a smooth constraint.

A typical objective is:

$$
\min_{\{W^{(k)}\}}\; \mathcal{L}_{\text{recon}} + \lambda_1\sum_{k=0}^{p}\|W^{(k)}\|_1 + \lambda_2 h(\cdot).
$$

Why useful:
- Explicitly separates lagged directional effects.
- Produces interpretable causal graphs for multivariate time series.

Limitation:
- Sensitive to regularization/threshold settings and may be expensive at larger scale.

In [ ]:
def plot_dag(adj, names, title, threshold=0.0):
    """Plot a weighted adjacency matrix as a network."""
    G = nx.DiGraph()
    G.add_nodes_from(names)
    for i, tgt in enumerate(names):
        for j, src in enumerate(names):
            w = adj[i, j]
            if w > threshold:
                G.add_edge(src, tgt, weight=float(w))
    pos = nx.spring_layout(G, seed=42)
    plt.figure(figsize=(8, 6))
    nx.draw_networkx(G, pos, with_labels=True, node_size=900, font_size=8, arrows=True)
    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.show()


def threshold_adjacency(w, threshold=0.35):
    a = (w >= threshold).astype(float)
    np.fill_diagonal(a, 0)
    return a

print("Plotting utilities ready.")


## Train DeepSCM (Fixed Graph)

This cell trains the **DeepSCM** model using a fixed adjacency matrix derived from correlation, then reports training progress.

In [ ]:
# Train DeepSCM (contemporaneous structural equations)
deep_scm = DeepSCM(
    lag=LAG, hidden=64,
    epochs=EPOCHS_SCM, batch_size=64, lr=1e-3,
    device=DEVICE, random_state=42,
)
deep_scm.fit(data_np)
print(f"[DeepSCM] final loss={deep_scm.history_['loss'][-1]:.4f}")

corr = np.abs(np.corrcoef(data_np.T))
np.fill_diagonal(corr, 0.0)
A_fixed = (corr > 0.25).astype(float)
plot_dag(A_fixed, VAR_NAMES, "Correlation heuristic (reference graph)")


## Train DECI and Inspect Learned Graph

This cell trains the **DECI** model, learns a soft causal adjacency matrix, thresholds it for interpretability, and visualizes the learned graph.

In [ ]:
# Train DECI (joint graph + structural equations)
deci = DECI(
    lag=LAG, hidden=64, lambda_dag=0.1,
    epochs=EPOCHS_DECI, batch_size=64, lr=1e-3,
    device=DEVICE, random_state=42,
)
deci.fit(data_np)
print(f"[DECI] final loss={deci.history_['loss'][-1]:.4f}")

A_soft = deci.adjacency_matrix()
A_bin = threshold_adjacency(A_soft, threshold=0.35)
plot_dag(A_soft, VAR_NAMES, "DECI Soft Adjacency")
plot_dag(A_bin, VAR_NAMES, "DECI Thresholded Adjacency")


## Intervention Analysis and ATE Estimation

`Intervention analysis` is a method used in causal inference to assess the effects of actively changing (intervening on) one or more variables within a system. It typically involves simulating or implementing a "do" operation (e.g., do(X = x)), which forces a variable to take on a specific value, and then evaluating how this change propagates through the system and affects other variables. This approach allows us to understand causal relationships and estimate quantities like the Average Treatment Effect (ATE), which measures the expected difference in an outcome variable due to the intervention.

In [ ]:
# Intervention + ATE example
source_name, target_name = "Energy", "Industrials"
source_idx = VAR_NAMES.index(source_name)
target_idx = VAR_NAMES.index(target_name)

x_contemp = data_np[:, :]  # contemporaneous slice for intervention demo
ate = deci.predict_ate(x_contemp, intervention_var=source_idx, n_samples=200)
print(f"Estimated ATE of do({source_name}) on {target_name}: {ate:.4f}")

pred_lo = deep_scm.intervene(x_contemp[:128], var_idx=source_idx, value=-1.0)
pred_hi = deep_scm.intervene(x_contemp[:128], var_idx=source_idx, value=1.0)
delta = (pred_hi[:, target_idx] - pred_lo[:, target_idx]).mean()
print(
    f"Explanation:\n"
    f"Estimated ATE of do({source_name}) on {target_name}: {ate:.4f}\n"
    f"DeepSCM intervention delta on {target_name}: {delta:.4f}"
)


## DYNOTEARS - DAG-Constrained Structure Learning

DYNOTEARS extends NOTEARS-style continuous optimization to **time-series causal discovery** by modeling lagged dependencies while enforcing acyclicity on the learned structure.

For lag order $p$, a common formulation is:

$$
X_t = W^{(0)}X_t + \sum_{k=1}^{p} W^{(k)}X_{t-k} + \epsilon_t,
$$

where:
- $W^{(0)}$ captures same-time (instantaneous) effects,
- $W^{(k)}$ captures directed lag-$k$ effects,
- acyclicity is imposed through a smooth NOTEARS-style constraint.

A typical optimization objective combines data fit, sparsity, and DAG regularization:

$$
\min_{\{W^{(k)}\}}\; \mathcal{L}_{\text{recon}} + \lambda_1\sum_{k=0}^{p}\|W^{(k)}\|_1 + \lambda_2 h(\cdot).
$$

Why it is useful in practice:
- Produces **interpretable directed graphs** with explicit lag structure.
- Provides DAG-regularized structure learning in a differentiable framework.
- Useful when structure-learning rigor is needed beyond purely predictive models.

Limitations:
- Sensitive to regularization and optimization hyperparameters.
- Can be expensive for high-dimensional or long-lag settings.

In [ ]:
# DynoTEARS — lag-resolved DAG discovery (pydeepcausalml.discovery)
dyno = DynoTEARS(
    lag=5, lambda_l1=0.02,
    epochs=EPOCHS_DYNO, batch_size=256, lr=3e-3,
    device=DEVICE, random_state=42,
)
dyno.fit(data_np)
print(f"[DynoTEARS] final loss={dyno.history_['loss'][-1]:.4f}")

A_dyno = dyno.get_adjacency(threshold=0.05)
W_agg = dyno.get_scores()
plot_dag(W_agg, VAR_NAMES, "DynoTEARS aggregated lag weights")
plot_dag(A_dyno, VAR_NAMES, "DynoTEARS thresholded adjacency")


## Graph Recovery Evaluation and Visual Comparison

This section ports the comparison/evaluation utilities from `Causal_Discovery_Time_Series.ipynb` and adapts them to variables available in this notebook. If a true graph is not available, metric computation is skipped while visual comparisons are still generated.

In [ ]:
# Graph comparison across methods available in this notebook
methods = {
    "DECI": deci.adjacency_matrix(),
    "DynoTEARS": dyno.get_adjacency(threshold=0.05),
}

fig, axes = plt.subplots(1, len(methods), figsize=(5 * len(methods), 4))
if len(methods) == 1:
    axes = [axes]
for ax, (name, mat) in zip(axes, methods.items()):
    sns.heatmap(mat, ax=ax, cmap="Blues", xticklabels=VAR_NAMES, yticklabels=VAR_NAMES)
    ax.set_title(name)
plt.tight_layout()
plt.show()

# Training dynamics
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for ax, (label, hist) in zip(
    axes,
    [("DeepSCM", deep_scm.history_), ("DECI", deci.history_), ("DynoTEARS", dyno.history_)],
):
    ax.plot(hist["loss"])
    ax.set_title(label)
    ax.set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
plt.tight_layout()
plt.show()


## Notes

- This notebook is a compact educational implementation inspired by DECI.
- The learned adjacency is soft during optimization; thresholding is for interpretability.
- Hyperparameters (`LAG`, graph threshold, DAG/sparsity penalties) affect discovered structure.
- For stronger causal claims, combine with domain priors and interventional validation.

## Summary and Conclusions

In this notebook, we explored structural causal models (SCMs) with a focus on learning and evaluating causal discovery in multivariate time series data. Using a deep learning-based SCM (akin to the DECI framework), we demonstrated steps to:

- Represent temporal data for causal discovery.
- Learn a causal graph ("who causes whom") using soft adjacency and regularization techniques.
- Visualize and threshold the learned DAG for interpretability.
- Perform intervention analysis by simulating "do-operations" on specific variables, enabling the estimation of treatment effects such as the Average Treatment Effect (ATE).

Through these steps, we illustrated how SCMs can move beyond correlation, allowing us to estimate the impact of interventions and support counterfactual reasoning in dynamical systems. The notebook highlights the importance of model assumptions, the tuning of hyperparameters (such as graph sparsity and DAG penalties), and the value of both observational and interventional data in causal inference.

**Key Takeaways:**
- Learning causal structure from time series is feasible and provides interpretable models of temporal influence.
- Deep SCMs enable interventional queries, supporting actionable insights in domains like economics and genomics.
- Thresholding and proper regularization are crucial for meaningful graphical outputs, but domain knowledge and further validation are needed for robust conclusions.

Overall, this tutorial provides a foundation for practitioners to begin using deep causal models for discovery and decision-making in complex real-world systems.

## Resources

- [Causality: Models, Reasoning and Inference (Judea Pearl)](https://bayes.cs.ucla.edu/BOOK-2K/)
- [Elements of Causal Inference (Peters, Janzing & Schölkopf)](https://mitpress.mit.edu/9780262037310/elements-of-causal-inference/)
- [DoWhy: An End-to-End Library for Causal Inference](https://github.com/py-why/dowhy)
- [DECI: Deep End-to-end Causal Inference](https://arxiv.org/abs/2202.02195)
- [An Introduction to Causal Inference (Judea Pearl, Statistical Surveys)](https://ftp.cs.ucla.edu/pub/stat_ser/r350.pdf)
- [DAGs with `networkx` and visualization](https://networkx.org/documentation/stable/reference/algorithms/dag.html)
- [A Tutorial on Learning with Bayesian Networks (Daphne Koller, Nir Friedman)](https://arxiv.org/abs/2002.00269)

*For further reading, see:*
- [The Book of Why (J. Pearl & D. Mackenzie)]
- [awesome-causal-inference: Curated list of causal inference resources](https://github.com/imirzadeh/awesome-causal-inference)